---
title: 2. 阅读 HybridML 仓库模型
---

# 2. 阅读 HybridML 仓库模型

这一章先用 Python 标准库读取模型 JSON，再观察 Theophylline 数据。这样可以先理解模型结构，不必立刻安装仓库的旧 TensorFlow 依赖。

从仓库根目录启动 JupyterLab，或确保 Notebook 所在目录的父目录中包含 `HybridML/ModelCreator.py`。

## 2.1 定位仓库并读取模型描述

HybridML 用 JSON 描述输入、节点、连线和输出。下面读取一个最小模型，以及一个由神经网络估计参数、再交给 ODE 节点的药代模型。

In [ ]:
import json
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    """向上寻找仓库根目录，避免依赖 Notebook 的固定工作目录。"""
    # 从当前位置逐级向父目录查找，适配“从仓库根目录启动”和“从 tutorials 启动”两种方式。
    # 例如当前目录是 repo/tutorials/chapters，依次会检查 chapters、tutorials、repo 等目录。
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "HybridML" / "ModelCreator.py").is_file():
            return candidate
    raise FileNotFoundError(
        "找不到 HybridML 仓库；请从仓库目录打开 JupyterLab。"
    )


repo_root = find_repo_root()
# Path 用 / 拼接路径组件，而不是手写盘符和斜杠，因此 Windows 与 macOS/Linux 都可用。
simple_path = repo_root / "demo" / "simple_model.json"
theoph_path = (
    repo_root
    / "demo" / "theoph" / "projects" / "theoph_demo"
    / "theoph_model_2_compartments.json"
)

# JSON 文件先读成文本，再解析为 Python 字典；例如 simple_model['name'] 就能取模型名。
simple_model = json.loads(simple_path.read_text(encoding="utf-8"))
theoph_model = json.loads(theoph_path.read_text(encoding="utf-8"))

print(f"仓库：{repo_root}")
print(f"最小示例：{simple_model['name']}")
print(f"药代示例：{theoph_model['name']}")

## 2.2 从节点和连接关系读模型

先看 JSON 中的输入和节点。`inputs` 是模型输入，节点的 `inputs`/`outputs` 是节点之间传递的连接器名称。它们不是 Pandas 列名，也不是神经网络层名称。

In [ ]:
def show_model(description):
    """打印模型图的关键结构，帮助把 JSON 映射成计算流程。"""
    print(f"\n模型：{description['name']}")
    print("模型输入：", [(item["id"], item["size"]) for item in description["inputs"]])
    # 按 JSON 中节点顺序逐个展示；连接器名字是图上的“线名”，不是表格字段名。
    for node in description["nodes"]:
        node_type = node["type"]
        if node_type == "nn":
            detail = [layer["size"] for layer in node["layers"]]
        elif node_type == "arithmetic":
            detail = node["expression"]
        else:
            detail = "机理/ODE 节点"
        print(
            f"  {node['id']} ({node_type}): "
            f"{node.get('inputs', [])} -> "
            f"{node.get('outputs', node.get('output', []))}; {detail}"
        )
    print("训练输出：", description["outputs"])
    print("额外可查看的中间量：", description.get("additional_outputs", []))


show_model(simple_model)
show_model(theoph_model)

### 药代模型的机理含义

Theophylline 示例把体重 `wt` 输入两个神经网络，得到正值参数 `ka` 和 `ke`。ODE 矩阵对应的状态方程为：

$$
\frac{d}{dt}\begin{bmatrix}x_1\\x_2\end{bmatrix}
=
\begin{bmatrix}-k_a&0\\k_a&-k_e\end{bmatrix}
\begin{bmatrix}x_1\\x_2\end{bmatrix},
\qquad x(0)=\begin{bmatrix}\mathrm{Dose}\\0\end{bmatrix}.
$$

在这个两状态模型中，状态 1（Python 索引 `1`）是模型输出的第二个隔室。JSON 用算术节点从完整 ODE 输出中选取它。读模型时要以状态方程和索引为准，不要仅凭节点名称推断生理含义。

**代入数字：** 暂取 `ka=0.5 h⁻¹`、`ke=0.1 h⁻¹`，某时刻 `x1=100`、`x2=20`（状态单位以模型定义为准）。则 `dx1/dt=-0.5×100=-50`，而 `dx2/dt=0.5×100-0.1×20=48`。第一室减少的 50 单位/小时大部分转入第二室，第二室同时以 2 单位/小时被消除，净增加 48。这里只用瞬时导数说明矩阵每一行代表什么，不是离散一步后的精确状态。

In [ ]:
# next(...) 找到第一个符合条件的节点；若 JSON 中不存在该类型，会明确抛出 StopIteration。
ode_node = next(node for node in theoph_model["nodes"] if node["type"] == "casadi_linear_ode")
print("ODE 系统矩阵：")
print(ode_node["system_matrix"])

output_node = next(
    node
    for node in theoph_model["nodes"]
    if node["type"] == "arithmetic" and "ode_output" in node.get("expression", "")
)
print("输出状态选择：", output_node["expression"])

## 2.3 看一眼数据

原始示例的数据在仓库中已经预处理为长表。下方按受试者查看记录数和时间范围。此文件是归一化数据，不要把其中的数值直接解释成原始临床单位。

In [ ]:
import pandas as pd


data_path = (
    repo_root / "demo" / "theoph" / "projects" / "theoph_demo"
    / "theoph_normalized.csv"
)
# CSV 读入 DataFrame；这里展示的是归一化后的数值，例如 0.5 不可直接当作 0.5 mg/L。
data = pd.read_csv(data_path)

display(data.head())
display(
    data.groupby("Subject").agg(
        records=("Time", "size"),
        first_time=("Time", "min"),
        last_time=("Time", "max"),
    ).head()
)

## 2.4 如何在原仓库中构建模型

HybridML 的公开入口是 `Project.create_model`。下面是运行原模型的入口示意；只有在单独安装并确认兼容仓库旧依赖后才取消注释。仓库要求 TensorFlow 2.3.1、TensorFlow Probability 0.11.1，并使用 CasADi ODE 节点；它不属于本教程的现代 Keras 环境。

```python
# from HybridML import Project
# model = Project.create_model(str(theoph_path))
# model.summary()
```

本章的目标是读懂“神经网络估计机理参数 → ODE 求解状态 → 选择模型输出”的结构。原始训练脚本还包含自定义优化器，不建议把它当作现代 Keras 的入门写法。